[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/module_06_confidentiality_attacks/Lab_6a_Membership_Inference_Attacks.ipynb)

# **Lab 6a: Membership Inference Attacks**

**Course:** Introduction to Data Security Pr.  
**Module 6:**  Confidentiality Attacks  
**Estimated Time:** 45 minutes

---

## Learning Objectives

By the end of this lab, you will understand:

1. **Membership Inference:** Determining if a specific sample was in a model's training set
2. **Privacy Leakage:** How overfitting causes models to "memorize" training data
3. **Attack Vectors:** Confidence-based membership inference
4. **Threat Model:** Adversary with black-box model access
5. **Defense Implications:** Differential privacy and other privacy-preserving techniques
6. **Real-World Impact:** GDPR/privacy implications of membership inference

## Table of Contents

1. [Threat Model: Privacy Leakage](#threat-model)
2. [Membership Inference Theory](#theory)
3. [Confidence-Based Attack](#confidence)
4. [Exercises](#exercises)

## Threat Model: Privacy Leakage <a id="threat-model"></a>

A **Membership Inference Attack** asks a simple but powerful question:

> *"Was sample **x** part of the training set?"*

The attack was formally introduced by [Shokri et al. (2017)](https://doi.org/10.48550/arXiv.1610.05820) and works by exploiting the **generalisation gap**: models tend to be more confident on training samples than on unseen ones.

### Real-World Scenarios:

<div align="center">

| Scenario | Attacker | Goal | Impact |
|----------|----------|------|--------|
| **Healthcare** | Patient | Did my data train this model? | Privacy violation |
| **Finance** | Customer | Is my transaction in training set? | Risk if model used for decisions |
| **Social Media** | User | Was my post used? | Consent violation |
| **ML Service** | Competitor | Extract training data info | Competitive intelligence |

</div>

### Threat Assumptions:

- **Black-Box Access:** Attacker can query model and observe outputs
- **Auxiliary Info:** Attacker may know some training data distribution
- **Model Characteristics:** Attacker knows model architecture but not training procedure

## Membership Inference Theory <a id="theory"></a>

- Why Does Membership Inference Work?

Overfitted models behave differently on training vs test data:

$$P(\text{correct prediction} | \text{training sample}) > P(\text{correct} | \text{test sample})$$

<img src="https://raw.githubusercontent.com/ELTE-DSED/Intro-Data-Security/main/module_06_confidentiality_attacks/images/bias_variance_tradeoff.png" width="80%" style="display:block; margin:auto;">

### Attack Principle:

<img src="https://raw.githubusercontent.com/ELTE-DSED/Intro-Data-Security/main/module_06_confidentiality_attacks/images/attack_principle_member_vs_nonmember.png" width="80%" style="display:block; margin:auto;">

- **Member (in training set):** Model predicts high confidence
- **Non-member (not in training set):** Model predicts lower confidence

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import Tuple, List
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load Breast Cancer Wisconsin dataset
data = load_breast_cancer()
X = data.data.astype(np.float32)
y = data.target.astype(np.int64)

# Split into candidate members and held-out non-members
X_train_full, X_temp, y_train_full, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Use a very small member set to encourage memorization and stronger leakage
member_train_size = 40
X_train, _, y_train, _ = train_test_split(
    X_train_full, y_train_full, train_size=member_train_size, random_state=42, stratify=y_train_full
)

# Standardize features using member-train statistics only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

train_data = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_data = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_data = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

input_dim = X_train.shape[1]
print(
    f"Training set (members): {len(train_data)}, Validation set: {len(val_data)}, "
    f"\nTest set (non-members): {len(test_data)}, Features: {input_dim}"
    f"\nClasses: {data.target_names.tolist()}"
)

## Model Architecture and Training

In [ ]:
class MLP(nn.Module):
    """MLP for tabular membership inference demo."""
    
    def __init__(self, input_dim: int, large: bool = False, dropout_rate: float = 0.0):
        super(MLP, self).__init__()
        if large:
            # Bigger model to encourage overfitting on the small training subset
            self.net = nn.Sequential(
                nn.Linear(input_dim, 512),
                nn.ReLU(),
                nn.Linear(512, 512),
                nn.ReLU(),
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 2)
            )
        else:
            # Compact model for the regularized baseline
            self.net = nn.Sequential(
                nn.Linear(input_dim, 32),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(32, 2)
            )

    def forward(self, x):
        return self.net(x)

In [ ]:
def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader = None, 
                epochs: int = 10, regularization: str = 'none') -> Tuple[List[float], List[float]]:
    """Train model with optional regularization."""
    model.to(device)
    
    if regularization == 'l2':
        # Use stronger weight decay on the regularized baseline
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.03)
    else:
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    
    criterion = nn.CrossEntropyLoss()
    train_losses = []
    val_losses = []
    
    with tqdm(range(epochs), desc="Training", unit="epoch") as pbar:
        for epoch in pbar:
            model.train()
            epoch_loss = 0.0
            for data, target in train_loader:
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation
            if val_loader is not None:
                model.eval()
                v_loss = 0.0
                with torch.no_grad():
                    for data, target in val_loader:
                        data, target = data.to(device), target.to(device)
                        output = model(data)
                        v_loss += criterion(output, target).item()
                avg_val_loss = v_loss / len(val_loader)
                val_losses.append(avg_val_loss)
                
                pbar.set_postfix(train_loss=f"{avg_train_loss:.4f}", val_loss=f"{avg_val_loss:.4f}")
            else:
                pbar.set_postfix(loss=f"{avg_train_loss:.4f}")
    
    return train_losses, val_losses

## PART 1: Training Models with Different Regularization

In [ ]:
print("\nTraining overfitted model (larger MLP, no regularization)...")
overfit_model = MLP(input_dim=input_dim, large=True).to(device)
overfit_losses, _ = train_model(overfit_model, train_loader, epochs=200, regularization='none')
print(f"Final training loss: {overfit_losses[-1]:.4f}")

In [ ]:
print("\nTraining regularized model (L2 = 0.03, Dropout = 0.5)...")
reg_model = MLP(input_dim=input_dim, large=False, dropout_rate=0.5).to(device)
reg_losses, val_losses = train_model(reg_model, train_loader, val_loader=val_loader, epochs=20, regularization='l2')
print(f"Final training loss: {reg_losses[-1]:.4f}")

### Evaluate on train and test

In [ ]:
def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, pred = torch.max(output.data, 1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * correct / total

In [ ]:
overfit_train_acc = evaluate(overfit_model, train_loader)
overfit_test_acc = evaluate(overfit_model, test_loader)

reg_train_acc = evaluate(reg_model, train_loader)
reg_test_acc = evaluate(reg_model, test_loader)

print(f"Model Performance:")
print(f"\nOverfitted Model (large, no reg):")
print(f"  Train accuracy: {overfit_train_acc:.2f}%")
print(f"  Test accuracy: {overfit_test_acc:.2f}%")
print(f"  Overfitting gap: {overfit_train_acc - overfit_test_acc:.2f}%")

print(f"\nRegularized Model (strong L2 + Dropout):")
print(f"  Train accuracy: {reg_train_acc:.2f}%")
print(f"  Test accuracy: {reg_test_acc:.2f}%")
print(f"  Overfitting gap: {reg_train_acc - reg_test_acc:.2f}%")

## PART 2: Confidence-Based Membership Inference Attack

In [ ]:
def get_confidence_scores(model: nn.Module, loader: DataLoader) -> np.ndarray:
    """Get max softmax confidence for all samples."""
    model.eval()
    confidences = []
    
    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            output = model(data)
            probs = torch.softmax(output, dim=1)
            conf = probs.max(dim=1)[0].cpu().numpy()
            confidences.extend(conf)
    
    return np.array(confidences)


def get_true_class_confidence(model: nn.Module, loader: DataLoader) -> np.ndarray:
    """Get probability assigned to the ground-truth class for each sample."""
    model.eval()
    true_confidences = []
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            probs = torch.softmax(model(data), dim=1)
            true_conf = probs[torch.arange(len(target), device=device), target].cpu().numpy()
            true_confidences.extend(true_conf)
    
    return np.array(true_confidences)

### Computing confidence scores

In [ ]:
# Max-softmax confidence
overfit_train_conf = get_confidence_scores(overfit_model, train_loader)
reg_train_conf = get_confidence_scores(reg_model, train_loader)
overfit_test_conf = get_confidence_scores(overfit_model, test_loader)
reg_test_conf = get_confidence_scores(reg_model, test_loader)

# True-class confidence (stronger signal for MI on this dataset)
overfit_train_true_conf = get_true_class_confidence(overfit_model, train_loader)
reg_train_true_conf = get_true_class_confidence(reg_model, train_loader)
overfit_test_true_conf = get_true_class_confidence(overfit_model, test_loader)
reg_test_true_conf = get_true_class_confidence(reg_model, test_loader)

print(f"Overfitted Model:")
print(f"  Max-softmax gap: {overfit_train_conf.mean() - overfit_test_conf.mean():.4f}")
print(
    f"  True-class confidence: train={overfit_train_true_conf.mean():.4f} ± {overfit_train_true_conf.std():.4f}, "
    f"test={overfit_test_true_conf.mean():.4f} ± {overfit_test_true_conf.std():.4f}"
)
print(f"  True-class gap (member signal): {overfit_train_true_conf.mean() - overfit_test_true_conf.mean():.4f}")

print(f"\nRegularized Model:")
print(f"  Max-softmax gap: {reg_train_conf.mean() - reg_test_conf.mean():.4f}")
print(
    f"  True-class confidence: train={reg_train_true_conf.mean():.4f} ± {reg_train_true_conf.std():.4f}, "
    f"test={reg_test_true_conf.mean():.4f} ± {reg_test_true_conf.std():.4f}"
)
print(f"  True-class gap (member signal): {reg_train_true_conf.mean() - reg_test_true_conf.mean():.4f}")

## Exercises

### Exercise 1: Model Size Impact
Train models of different sizes and measure membership inference success. Does larger model = more vulnerable? Why?

### Exercise 2: Training Set Size
Train models on different training set sizes. Does more data help defend against MI?

### Exercise 3: Regularization Comparison
Compare different regularization techniques (L1, L2, Dropout, Batch normalization). Which provides best protection against membership inference?